# 03 · Fishing Pressure Inside Protected Natura 2000 Zones

**Team Vega · SODAS Hackathon 2026 · University of Copenhagen**

Denmark's marine [Natura 2000](https://en.wikipedia.org/wiki/Natura_2000) areas are
protected habitats (reefs, sandbanks, spawning grounds). This notebook asks a pointed
question:

> **How much fishing activity is happening *inside* these protected zones — and which
> vessels are responsible?**

We do a **spatial join** between the fishing pings and the official Natura 2000 polygons,
classify each ping by behaviour (using the same trawling-speed heuristic as notebook 02),
and rank the vessels with the most pings inside protected areas.

**Inputs:**
- `../data/N2000/np3_2022_marine_kortlaeg_2004_2018.shp` — Natura 2000 marine mapping (download separately, see README)
- `../data/raw/aisdk-<year>-1h.parquet` — raw AIS

> *Headline finding (2022): of ~651k fishing pings that fell inside protected zones, ~12%
> showed the slow, steady speed signature of active trawling.*

In [ ]:
import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

RAW_DIR = "../data/raw"
N2000_DIR = "../data/N2000"
ZONES_SHAPEFILE = f"{N2000_DIR}/np3_2022_marine_kortlaeg_2004_2018.shp"

## 1. Load the Natura 2000 protected zones

The shapefile is published in the Danish national grid (EPSG:25832). We reproject it to
WGS84 (EPSG:4326) so it lines up with the GPS coordinates in the AIS feed, and grab its
bounding box to filter the AIS pings cheaply later.

In [ ]:
print("Loading Natura 2000 mapping...")
zones = gpd.read_file(ZONES_SHAPEFILE)

# Some shapefiles ship without an embedded CRS — set the Danish national grid manually.
if zones.crs is None:
    print("CRS missing - setting it manually to EPSG:25832")
    zones.set_crs(epsg=25832, inplace=True)

zones = zones.to_crs(epsg=4326)
minx, miny, maxx, maxy = zones.total_bounds
print(f"{len(zones)} protected polygons, bounding box: "
      f"({minx:.2f}, {miny:.2f}) -> ({maxx:.2f}, {maxy:.2f})")

## 2. Load AIS and restrict to the zones' bounding box

We only need a handful of columns. Filtering to the bounding box *before* the expensive
spatial join throws away the vast majority of pings (everything far from any protected
area) and keeps the join fast.

In [ ]:
print("Loading AIS data (parquet)...")
columns = ["# Timestamp", "MMSI", "Latitude", "Longitude", "SOG", "Ship type", "Name"]
df_ais = pd.read_parquet(f"{RAW_DIR}/aisdk-2022-1h.parquet", columns=columns)

print("Filtering pings to the protected-area bounding box...")
df_focus = df_ais[
    df_ais["Latitude"].between(miny, maxy)
    & df_ais["Longitude"].between(minx, maxx)
].copy()
print(f"{len(df_focus):,} pings remain inside the bounding box")

## 3. Spatial join — which pings fall *inside* a protected polygon?

We turn the surviving pings into point geometries and use a `within` spatial join. Every
row that comes back is a broadcast that physically occurred inside a Natura 2000 zone.

In [ ]:
print(f"Converting {len(df_focus):,} pings to geometry...")
gdf_ais = gpd.GeoDataFrame(
    df_focus,
    geometry=gpd.points_from_xy(df_focus["Longitude"], df_focus["Latitude"]),
    crs="EPSG:4326",
)

print("Checking for overlap with protected areas (spatial join)...")
inside = gpd.sjoin(gdf_ais, zones, predicate="within")
print(f"Done. Found {len(inside):,} pings inside protected areas.")

inside.to_csv("../data/natura2000_incursions_2022.csv", index=False)
print("Saved results to data/natura2000_incursions_2022.csv")

## 4. Classify behaviour by speed

A vessel merely *transiting* a zone at speed is very different from one slowly trawling
inside it. We bucket each ping by speed over ground (`SOG`), treating the 2–5 knot band as
**potential active fishing**.

In [ ]:
def categorize_speed(sog: float) -> str:
    if sog < 0.5:
        return "Stationary"
    if 2 <= sog <= 5:
        return "Potential Fishing"
    if sog > 10:
        return "Transit"
    return "Other"


inside["Activity"] = inside["SOG"].apply(categorize_speed)
print(inside["Activity"].value_counts())

## 5. How much activity is suspicious, and who is responsible?

We quantify the share of in-zone pings that look like active fishing, then rank the vessels
(by `MMSI`) that contribute the most of it.

In [ ]:
fishing_pings = (inside["Activity"] == "Potential Fishing").sum()
total_pings = len(inside)
print(f"Total pings in protected zones: {total_pings:,}")
print(f"Potential fishing incidents:    {fishing_pings:,}")
print(f"Share of suspicious activity:   {fishing_pings / total_pings * 100:.1f}%")

In [ ]:
# Top vessels by number of potential-fishing pings inside protected zones.
fishing_only = inside[inside["Activity"] == "Potential Fishing"]
top_fishers = fishing_only["MMSI"].value_counts().head(5)

print("--- Top 5 vessels by potential-fishing pings inside protected zones ---")
for mmsi, count in top_fishers.items():
    name_lookup = pd.read_parquet(
        f"{RAW_DIR}/aisdk-2022-1h.parquet",
        filters=[("MMSI", "==", mmsi)],
        columns=["Name"],
    )
    ship_name = name_lookup["Name"].iloc[0] if not name_lookup.empty else "Unknown"
    print(f"Name: {ship_name} | MMSI: {mmsi} | Fishing pings: {count}")

## 6. Map the pressure

Finally we plot the potential-fishing pings (red) over the protected zones on an
OpenStreetMap basemap. The clusters of red along the coasts and around reefs are exactly
the hotspots a marine inspector would want to prioritise — this is the map featured in the
project README and presentation.

In [ ]:
fishing_pings_gdf = inside[inside["Activity"] == "Potential Fishing"]

fig, ax = plt.subplots(figsize=(12, 15))

# Protected zones as translucent green outlines.
zones.to_crs(epsg=3857).plot(
    ax=ax, color="green", alpha=0.3, edgecolor="darkgreen", linewidth=1
)

# Potential fishing pings as small red dots.
fishing_pings_gdf.to_crs(epsg=3857).plot(
    ax=ax, color="red", markersize=0.2, alpha=0.4, label="Potential Fishing"
)

cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, crs="EPSG:3857")
ax.set_title("Fishing Pressure in Danish Natura 2000 Zones (2022)", fontsize=18)
ax.set_axis_off()
plt.legend(loc="upper right", markerscale=20)
plt.show()

---
### Where to take this next
- Run the same join across 2021–2026 to see whether in-zone fishing pressure is rising or
  falling over time.
- Cross-reference top vessels against the public Danish vessel register.
- Weight by *time spent* in a zone rather than raw ping counts.

*Team Vega — SODAS Hackathon 2026, University of Copenhagen.*